# RLM (Recursive Language Models) - Colab Quickstart

This notebook demonstrates how to use RLM with Google Colab's built-in AI library.
**No API key required!**

RLM enables language models to recursively call themselves through a REPL environment,
allowing for complex reasoning and code execution.

Paper: [arXiv:2512.24601](https://arxiv.org/abs/2512.24601)

## Step 1: Check Python Version

RLM requires Python 3.11 or higher.

In [ ]:
import sys
print(f"Python version: {sys.version}")

if sys.version_info < (3, 11):
    print("WARNING: Python 3.11+ is required!")
    print("Go to Runtime > Change runtime type > select Python 3.11")
else:
    print("Python version OK!")

## Step 2: Install RLM with Colab AI Support

**Important:** The `colab_ai` backend is available in this fork. 
If you previously installed from the official repo, you need to reinstall.

In [ ]:
# Uninstall existing rlm (if any) and install from fork with colab_ai support
!pip uninstall -y rlm 2>/dev/null || true
!pip install -q git+https://github.com/jaeeyoonpark/rlm.git@claude/colab-execution-setup-MpxnF

print("RLM with colab_ai backend installed!")

# Check if restart is needed (only if rlm was already imported)
import sys
if 'rlm' in sys.modules:
    print("\n⚠️  rlm was already imported. Please restart runtime:")
    print("   Runtime > Restart session, then run from Step 3.")

## Step 3: Verify Colab AI is Available

In [ ]:
try:
    from google.colab import ai
    print("google.colab.ai is available!")
    
    # Quick test
    response = ai.generate_text("Say 'Hello RLM!' and nothing else.")
    print(f"Test response: {response}")
except ImportError:
    print("ERROR: google.colab.ai is not available.")
    print("Make sure you are running this in Google Colab.")

## Step 4: Run RLM with Colab AI Backend

Now let's use RLM with the `colab_ai` backend. This uses Gemini models through Colab's built-in AI library.

In [ ]:
from rlm import RLM

# Initialize RLM with Colab AI backend
rlm = RLM(
    backend="colab_ai",
    backend_kwargs={
        "model_name": "gemini-2.5-flash-lite",  # free tier model
    },
    environment="local",
    environment_kwargs={},
    max_depth=1,
    max_iterations=10,
    verbose=True,
)

print("RLM initialized with Colab AI backend!")

## Step 5: Test with a Simple Task

In [ ]:
# Simple computation task
result = rlm.completion(
    "Calculate the first 10 Fibonacci numbers and print each one on a new line."
)

print("\n" + "="*50)
print("FINAL RESULT:")
print("="*50)
print(result.response)

## Step 6: Run with Logger for Analysis

Let's run a task with logging enabled so we can analyze the execution trace.

In [ ]:
from rlm import RLM
from rlm.logger import RLMLogger

# Create a logger to capture execution trace
logger = RLMLogger(log_dir="/tmp/rlm_logs", file_name="analysis")

# Initialize RLM with logger
rlm_with_logger = RLM(
    backend="colab_ai",
    backend_kwargs={"model_name": "gemini-2.5-flash-lite"},
    environment="local",
    max_depth=1,
    max_iterations=10,
    logger=logger,
    verbose=True,
)

# Run a task
result = rlm_with_logger.completion("""
Create a list of 20 random integers between 1 and 100.
Then calculate and print:
1. The mean
2. The median
3. The standard deviation
4. The min and max values
""")

print("\n" + "="*50)
print("FINAL RESULT:")
print("="*50)
print(result.response)
print(f"\nLog file: {logger.log_file_path}")

## Step 7: Analyze the Execution Trace

Now let's analyze what happened during the RLM execution.

In [ ]:
import json

# Load the log file
iterations = []
with open(logger.log_file_path, 'r') as f:
    for line in f:
        entry = json.loads(line)
        if entry.get('type') == 'iteration':
            iterations.append(entry)

print(f"Total iterations: {len(iterations)}")
print("="*60)

### 7.1 Root-LM Prompts and Responses

What prompts did the Root-LM receive and how did it respond?

In [ ]:
for i, iteration in enumerate(iterations):
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration['iteration']}")
    print(f"{'='*60}")
    
    # Show the prompt (last user message if it's a list)
    prompt = iteration['prompt']
    if isinstance(prompt, list):
        # Find the last user message
        for msg in reversed(prompt):
            if msg.get('role') == 'user':
                print(f"\n📥 PROMPT TO ROOT-LM:")
                print("-"*40)
                content = msg.get('content', '')
                # Truncate if too long
                if len(content) > 500:
                    print(content[:500] + "...[truncated]")
                else:
                    print(content)
                break
    else:
        print(f"\n📥 PROMPT TO ROOT-LM:")
        print("-"*40)
        print(prompt[:500] if len(str(prompt)) > 500 else prompt)
    
    # Show Root-LM response
    print(f"\n📤 ROOT-LM RESPONSE:")
    print("-"*40)
    response = iteration['response']
    if len(response) > 800:
        print(response[:800] + "...[truncated]")
    else:
        print(response)

### 7.2 Code Blocks Executed

What code did the Root-LM generate and execute?

In [ ]:
for iteration in iterations:
    code_blocks = iteration.get('code_blocks', [])
    if not code_blocks:
        continue
        
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration['iteration']} - CODE BLOCKS")
    print(f"{'='*60}")
    
    for j, block in enumerate(code_blocks):
        print(f"\n🐍 CODE BLOCK {j+1}:")
        print("-"*40)
        print(block['code'])
        
        result = block.get('result', {})
        
        # Show stdout
        stdout = result.get('stdout', '')
        if stdout:
            print(f"\n📺 STDOUT:")
            print("-"*40)
            print(stdout[:500] if len(stdout) > 500 else stdout)
        
        # Show stderr if any
        stderr = result.get('stderr', '')
        if stderr:
            print(f"\n⚠️ STDERR:")
            print("-"*40)
            print(stderr[:300] if len(stderr) > 300 else stderr)

### 7.3 Sub-LM Calls (Recursive Calls)

Did the code make any recursive calls to the LM? These happen when code uses `rlm.completion()` within the REPL.

In [ ]:
sub_lm_calls_found = False

for iteration in iterations:
    code_blocks = iteration.get('code_blocks', [])
    
    for j, block in enumerate(code_blocks):
        result = block.get('result', {})
        rlm_calls = result.get('rlm_calls', [])
        
        if rlm_calls:
            sub_lm_calls_found = True
            print(f"\n{'='*60}")
            print(f"ITERATION {iteration['iteration']} - BLOCK {j+1} - SUB-LM CALLS")
            print(f"{'='*60}")
            
            for k, call in enumerate(rlm_calls):
                print(f"\n🔄 SUB-LM CALL {k+1}:")
                print(f"   Model: {call.get('root_model', 'unknown')}")
                print(f"   Execution time: {call.get('execution_time', 0):.2f}s")
                
                print(f"\n   📥 SUB-LM PROMPT:")
                print("-"*40)
                sub_prompt = call.get('prompt', '')
                if len(str(sub_prompt)) > 400:
                    print(str(sub_prompt)[:400] + "...[truncated]")
                else:
                    print(sub_prompt)
                
                print(f"\n   📤 SUB-LM RESPONSE:")
                print("-"*40)
                sub_response = call.get('response', '')
                if len(sub_response) > 400:
                    print(sub_response[:400] + "...[truncated]")
                else:
                    print(sub_response)

if not sub_lm_calls_found:
    print("No sub-LM calls were made in this execution.")
    print("Sub-LM calls occur when the generated code calls rlm.completion() recursively.")

### 7.4 Execution Summary

In [ ]:
# Summary statistics
total_code_blocks = sum(len(it.get('code_blocks', [])) for it in iterations)
total_sub_calls = sum(
    len(block.get('result', {}).get('rlm_calls', []))
    for it in iterations
    for block in it.get('code_blocks', [])
)
total_time = sum(it.get('iteration_time', 0) for it in iterations)

print("="*60)
print("EXECUTION SUMMARY")
print("="*60)
print(f"📊 Total iterations:     {len(iterations)}")
print(f"🐍 Total code blocks:    {total_code_blocks}")
print(f"🔄 Total sub-LM calls:   {total_sub_calls}")
print(f"⏱️  Total iteration time: {total_time:.2f}s")
print(f"✅ Final answer:         {result.response[:100]}..." if len(result.response) > 100 else f"✅ Final answer: {result.response}")

## Available Models

With Colab AI, you can use the following Gemini models:

| Model | Description |
|-------|-------------|
| `gemini-2.5-flash-lite` | Free tier, lightweight (default) |
| `gemini-2.5-flash` | Free tier, more capable |

Free tier users have monthly usage limits.

## Tips

1. **Python Version**: Make sure you're using Python 3.11+
2. **Usage Limits**: Free tier has monthly limits, Pro users get more
3. **Local Environment**: The `local` environment runs code directly in Colab
4. **Verbose Mode**: Set `verbose=True` to see the execution trace